In [1]:
import pandas as pd
import mediapipe as mp
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import numpy as np


In [2]:
def calculate_angle(a,b,c):
    radians = np.arctan2(c.y-b.y, c.x-b.x) - np.arctan2(a.y-b.y, a.x-b.x)
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle


In [3]:

# 1. Load the CSV File
csv_file = "landmarks_output.csv"  # Path to your CSV file
df = pd.read_csv(csv_file)

# 2. Prepare the Data
# The last column in the CSV is our class (lunges or nonlunges), the remaining columns are the features
X = df.iloc[:, 1:-1].values  # All landmark data (X, Y, Z, Visibility)
y = df.iloc[:, -1].values  # Class labels (lunges or nonlunges)

# Convert the class labels (lunges and nonlunges) into numeric values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# 3. Train the Random Forest Model
clf = RandomForestClassifier(n_estimators=10, random_state=42)
clf.fit(X_train, y_train)

# 4. Make Predictions with a Test Image
# Initialize the Mediapipe Pose module
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1, enable_segmentation=False)


In [4]:
# Load a test image (test.jpg)
test_image_path = "testimages/test7.jpg"  # Specify the path to your test image here
test_image = cv2.imread(test_image_path)

# Convert the image to RGB (Mediapipe works in RGB format)
test_image_rgb = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)

# Perform pose estimation with the test image
results = pose.process(test_image_rgb)

# If landmarks are detected, make predictions
if results.pose_landmarks:
    # Convert landmark data into a single row
    test_row = []
    test_row.append(calculate_angle(results.pose_landmarks.landmark[12],results.pose_landmarks.landmark[24],results.pose_landmarks.landmark[26]))
    test_row.append(calculate_angle(results.pose_landmarks.landmark[24],results.pose_landmarks.landmark[26],results.pose_landmarks.landmark[28]))
    test_row.append(calculate_angle(results.pose_landmarks.landmark[11],results.pose_landmarks.landmark[23],results.pose_landmarks.landmark[25]))
    test_row.append(calculate_angle(results.pose_landmarks.landmark[23],results.pose_landmarks.landmark[25],results.pose_landmarks.landmark[27]))



        

    # Make predictions with the Random Forest model
    prediction = clf.predict([test_row])

    # Output the predicted class (lunges or nonlunges)
    predicted_class = label_encoder.inverse_transform(prediction)
    print(f"Pose prediction in the test image: {predicted_class[0]}")
else:
    print("No pose landmarks detected in the test image.")

Pose prediction in the test image: rlunges


In [ ]:

# Close the Pose module
pose.close()